## Подготовка

### Импорт

In [1]:
import openrouteservice
import folium
import geopandas as gpd
import pandas as pd
import json
import time
import os
from datetime import datetime, timedelta
from shapely.geometry import shape, Polygon, MultiPolygon
from tqdm.notebook import tqdm
from IPython.display import display, HTML

### Конфигурация

In [2]:
ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImYwNTUxYjkyYTcxOTQxNjc4YTU5MmEwMjVhOTljMDc1IiwiaCI6Im11cm11cjY0In0="

# Лимиты API
RATE_LIMIT_PER_MINUTE = 20
RATE_LIMIT_PER_DAY = 500

# Параметры изохрон
PROFILE = "foot-walking"  # driving-car, driving-hgv, cycling-regular, foot-walking и др.
RANGE_SECONDS = [300, 600, 900]  # 5, 10, 15 минут
# Альтернативно в метрах:
RANGE_TYPE = "distance"
RANGE_METERS = [500]
# RANGE_TYPE = "time"
# RANGE_VALUES !!!!
# Файл-счётчик для отслеживания дневного лимита
COUNTER_FILE = "ors_daily_counter.json"

### Rate Limiter (Ограничетель минутного/суточного подсчета)

In [3]:
from datetime import datetime, timedelta, timezone

class ORSRateLimiter:

    def __init__(
        self,
        per_minute: int = RATE_LIMIT_PER_MINUTE,
        per_day: int = RATE_LIMIT_PER_DAY,
        counter_file: str = COUNTER_FILE,
    ):
        self.per_minute = per_minute
        self.per_day = per_day
        self.counter_file = counter_file
        self.minute_timestamps: list[float] = []
        self._load_daily_counter()

    def _load_daily_counter(self):
        if os.path.exists(self.counter_file):
            with open(self.counter_file, "r") as f:
                data = json.load(f)
            saved_date = data.get("date", "")
            today = datetime.now(timezone.utc).strftime("%Y-%m-%d")  # ✅
            if saved_date == today:
                self.daily_count = data.get("count", 0)
            else:
                self.daily_count = 0
        else:
            self.daily_count = 0

    def _save_daily_counter(self):
        data = {
            "date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),  # ✅
            "count": self.daily_count,
        }
        with open(self.counter_file, "w") as f:
            json.dump(data, f)

    def wait_if_needed(self):
        if self.daily_count >= self.per_day:
            now = datetime.now(timezone.utc)  # ✅
            midnight = (now + timedelta(days=1)).replace(
                hour=0, minute=0, second=0, microsecond=0
            )
            wait_sec = (midnight - now).total_seconds() + 5
            print(
                f"⛔ Дневной лимит ({self.per_day}) исчерпан. "
                f"Ждём до полуночи UTC (~{wait_sec/3600:.1f} ч)."
            )
            raise RuntimeError(
                f"Дневной лимит {self.per_day} запросов исчерпан. "
                f"Продолжение возможно после {midnight.isoformat()}"
            )

        now = time.time()
        self.minute_timestamps = [
            t for t in self.minute_timestamps if now - t < 60
        ]
        if len(self.minute_timestamps) >= self.per_minute:
            oldest = self.minute_timestamps[0]
            sleep_time = 60 - (now - oldest) + 0.5
            if sleep_time > 0:
                print(
                    f"⏳ Минутный лимит ({self.per_minute}/мин). "
                    f"Пауза {sleep_time:.1f} сек..."
                )
                time.sleep(sleep_time)

    def register_request(self):
        self.minute_timestamps.append(time.time())
        self.daily_count += 1
        self._save_daily_counter()

    @property
    def remaining_today(self) -> int:
        return max(0, self.per_day - self.daily_count)

    def reset_daily_counter(self):
        self.daily_count = 0
        self._save_daily_counter()
        print("✅ Дневной счётчик сброшен.")

    def status(self):
        print(f"📊 Запросов сегодня: {self.daily_count}/{self.per_day}")
        print(f"   Осталось: {self.remaining_today}")


# Создаём экземпляр
rate_limiter = ORSRateLimiter()
rate_limiter.status()

📊 Запросов сегодня: 0/500
   Осталось: 500


### Функция расчёта изохрон

In [4]:
# Инициализация клиента
client = openrouteservice.Client(key=ORS_API_KEY)


def compute_isochrone(
    lon: float,
    lat: float,
    profile: str = PROFILE,
    range_values: list = RANGE_METERS,
    range_type: str = RANGE_TYPE,
    point_id: str = None,
) -> dict | None:
    """
    Вычисляет изохрону для одной точки.
    Возвращает GeoJSON-подобный словарь или None при ошибке.
    """
    rate_limiter.wait_if_needed()

    try:
        result = client.isochrones(
            locations=[[lon, lat]],
            profile=profile,
            range=range_values,
            range_type=range_type,
            attributes=["area", "total_pop"],  # если доступно
        )
        rate_limiter.register_request()

        # Добавляем point_id в properties каждого feature
        if point_id is not None:
            for feat in result.get("features", []):
                feat["properties"]["point_id"] = point_id
                feat["properties"]["source_lon"] = lon
                feat["properties"]["source_lat"] = lat

        return result

    except openrouteservice.exceptions.ApiError as e:
        rate_limiter.register_request()  # Считаем в лимит даже при ошибке
        print(f"❌ API ошибка для точки ({lat}, {lon}): {e}")
        return None
    except Exception as e:
        print(f"❌ Ошибка для точки ({lat}, {lon}): {e}")
        return None

In [5]:
RANGE_METERS

[500]

## Расчет

### Загрузка исходного слоя остановок

In [6]:
gdf = gpd.read_file(r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\routes\yandex_izhevsk_28409.gpkg", layer='stops_0')
gdf = gdf.to_crs(4326)
gdf = gdf[gdf['ya_line_id'].str.contains('10032')] #заменить в зависимости от маршрута

In [7]:
points = pd.DataFrame({
    "id": gdf['ya_stop_id'],
    "lat": gdf.geometry.y,
    "lon": gdf.geometry.x,
})

In [8]:
print(f"📍 Всего точек: {len(points)}")
print(f"📊 Осталось запросов сегодня: {rate_limiter.remaining_today}")

if len(points) > rate_limiter.remaining_today:
    print(
        f"⚠️  Точек ({len(points)}) больше, чем осталось запросов "
        f"({rate_limiter.remaining_today}). Часть не будет обработана."
    )

display(points.head(10))

📍 Всего точек: 32
📊 Осталось запросов сегодня: 500


,id,lat,lon
29,stop__9884251,56.849653,53.225398
30,stop__9884253,56.855390,53.223106
31,stop__9884256,56.860315,53.221212
32,stop__9884257,56.865740,53.223584
57,4373543286,56.864659,53.222913
58,stop__9884255,56.860615,53.220588
59,stop__9884254,56.855655,53.222286
60,stop__9884252,56.850349,53.224523
267,stop__9884249,56.841074,53.227885
286,stop__9884250,56.842149,53.227970


### Пакетный расчёт с прогресс-баром

In [9]:
all_features = []
errors = []
results_cache_file = "isochrones_cache.geojson"

# Проверяем, есть ли кэш незавершённого расчёта
already_computed = set()
if os.path.exists(results_cache_file):
    cached_gdf = gpd.read_file(results_cache_file)
    if "point_id" in cached_gdf.columns:
        already_computed = set(cached_gdf["point_id"].unique())
        # Загружаем ранее посчитанные features
        all_features = json.loads(cached_gdf.to_json())["features"]
        print(
            f"📂 Загружено из кэша: {len(already_computed)} точек "
            f"({len(all_features)} полигонов)"
        )

# Фильтруем то, что уже посчитано
points_todo = points[~points["id"].astype(str).isin(already_computed)]
print(f"🔄 Осталось посчитать: {len(points_todo)} точек")

# ---- Основной цикл ----
for idx, row in tqdm(
    points_todo.iterrows(), total=len(points_todo), desc="Изохроны"
):
    point_id = str(row["id"])
    lat, lon = row["lat"], row["lon"]

    if rate_limiter.remaining_today <= 0:
        print(f"\n⛔ Дневной лимит исчерпан. Остановка на точке '{point_id}'.")
        print("   Перезапустите эту ячейку завтра — кэш сохранён.")
        break

    result = compute_isochrone(
        lon=lon, lat=lat, point_id=point_id
    )

    if result and "features" in result:
        all_features.extend(result["features"])
        # Инкрементально сохраняем кэш
        _save_gdf = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")
        _save_gdf.to_file(results_cache_file, driver="GeoJSON")
    else:
        errors.append({"id": point_id, "lat": lat, "lon": lon})

print(f"\n✅ Готово! Полигонов: {len(all_features)}, ошибок: {len(errors)}")
rate_limiter.status()

if errors:
    print("\n❌ Точки с ошибками:")
    display(pd.DataFrame(errors))

📂 Загружено из кэша: 688 точек (688 полигонов)
🔄 Осталось посчитать: 2 точек


Изохроны:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Готово! Полигонов: 690, ошибок: 0
📊 Запросов сегодня: 2/500
   Осталось: 498


### Сборка GeoDataFrame с изохронами

In [10]:
if all_features:
    gdf_iso = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")

    # Приводим значения диапазонов к минутам (если time)
    if RANGE_TYPE == "time" and "value" in gdf_iso.columns:
        gdf_iso["range_minutes"] = gdf_iso["value"] / 60

    print(f"📐 Итоговый GeoDataFrame: {len(gdf_iso)} строк")
    display(gdf_iso.head())
else:
    print("⚠️ Нет данных для построения.")
    gdf_iso = gpd.GeoDataFrame()

📐 Итоговый GeoDataFrame: 690 строк


,geometry,group_index,value,area,total_pop,point_id,source_lon,source_lat,center
0,"POLYGON ((53.1941 56.84406, 53.1977 56.84091, ...",0,500.0,536252.63,4761.0,stop__9884354,53.201572,56.843624,None
1,"POLYGON ((53.19336 56.83974, 53.19643 56.83776...",0,500.0,500412.80,2763.0,stop__9884378,53.200770,56.839490,None
2,"POLYGON ((53.17644 56.83143, 53.17841 56.83056...",0,500.0,417812.07,1575.0,stop__9884383,53.185158,56.830829,None
3,"POLYGON ((53.17711 56.82762, 53.18093 56.82514...",0,500.0,430162.51,1206.0,stop__9884414,53.184843,56.827684,None
4,"POLYGON ((53.17123 56.8167, 53.17121 56.81634,...",0,500.0,545037.73,4317.0,stop__9884444,53.177619,56.817411,None


## Итог

### Экспорт результатов

In [12]:
if not gdf_iso.empty:
    # # GeoJSON
    # gdf_iso.to_file("isochrones_result.geojson", driver="GeoJSON")
    # print("💾 isochrones_result.geojson")

    # GeoPackage (удобнее для QGIS)
    gdf_iso.to_file(r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\isochrones_result_0.gpkg", driver="GPKG", layer="isochrones")
    print("💾 isochrones_result.gpkg")

    # Shapefile
    # gdf_iso.to_file("isochrones_result.shp")

    # Площади в км²
    if gdf_iso.crs and gdf_iso.crs.is_geographic:
        gdf_proj = gdf_iso.to_crs(epsg=3857)
        gdf_iso["area_km2"] = gdf_proj.geometry.area / 1e6

    # Сводная таблица
    if "point_id" in gdf_iso.columns and "value" in gdf_iso.columns:
        summary = gdf_iso.pivot_table(
            index="point_id",
            columns="value",
            values="area_km2",
            aggfunc="first",
        )
        if RANGE_TYPE == "time":
            summary.columns = [f"{int(c/60)} мин" for c in summary.columns]
        print("\n📊 Площади изохрон (км²):")
        display(summary.round(2))
        summary.to_csv("isochrones_summary.csv")
        print("💾 isochrones_summary.csv")

💾 isochrones_result.gpkg

📊 Площади изохрон (км²):


value,500.0
point_id,
1543164495,1.43
1543164559,1.04
1543164597,1.11
1543164637,1.89
1543164687,1.48
...,...
stop__9884544,1.83
stop__9884545,1.94
stop__9914195,1.84


💾 isochrones_summary.csv


### Утилиты

In [66]:
# Проверить статус лимитов
# rate_limiter.status()

# Сбросить дневной счётчик вручную (если знаете, что новые сутки)
# rate_limiter.reset_daily_counter()

# Удалить кэш для полного пересчёта
# os.remove("isochrones_cache.geojson")

✅ Дневной счётчик сброшен.


C:\Users\WSY\AppData\Local\Temp\ipykernel_16500\603815228.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date": datetime.utcnow().strftime("%Y-%m-%d"),
